# OpenClaw + DiffusionGemma (NVFP4) on a Colab L4 — chat in the notebook UI

Notebook counterpart of the validated bash master
`bin/colab_openclaw_diffusiongemma.sh --gpu L4 --config configs/diffusiongemma_nvfp4.json`
(which drives `remote/remote_colab_openclaw_diffusiongemma.py` on a Colab L4 headlessly via the
`colab` CLI). Here the **same bootstrap → prompt phases** run as in-cell steps on **your own L4
runtime**, so you can chat with OpenClaw right here.

**Setup:** `Runtime → Change runtime type → L4 GPU` → Save, then `Runtime → Run all`. **Keep this
tab open** — your browser is the runtime heartbeat (this is also why there's no ~10-min keep-alive
cap here, unlike the headless CLI path). First run installs vLLM + downloads the ~13 GB NVFP4
checkpoint + loads it (~10–12 min, one time).

> **L4 required.** DiffusionGemma-26B-A4B-NVFP4 needs ~24 GB VRAM (Colab Pro / compute units). It
> will **not** fit on a free T4.

## 📋 Briefing — what & why

**Goal.** Stand up the tested DiffusionGemma stack — vLLM serving `RedHatAI/diffusiongemma-26B-A4B-it-NVFP4` on loopback, an
OpenClaw Gateway pointed at it — and chat with it from the notebook, with no public tunnel.

**Why vLLM here (not llama.cpp).** DiffusionGemma is a block-diffusion model shipped as **NVFP4**;
vLLM loads it on an L4 (Ada sm_89) via the **Marlin FP4 weight-only** fallback (NVFP4 is
Blackwell-native, but the weight-only path works on Ada). `DiffusionGemmaForBlockDiffusion` needs
`--trust-remote-code`. (The llama.cpp path in `openclaw_chat_colab.ipynb` is the fee-free T4 floor
model; this is the heavyweight target on an L4.)

**Containment.** Everything is loopback: vLLM on `127.0.0.1:8000`, OpenClaw gateway on
`127.0.0.1:18789`. Nothing is exposed off the VM.

**Phase map (this notebook ⟷ the `bin/` master).**

| Notebook cell | Master phase (remote action) |
|---|---|
| 1 install | `bootstrap` → `install_vllm` + OpenClaw install |
| 2 serve | `bootstrap` → `start_vllm` (serve :8000, NVFP4 on L4) |
| 3 onboard + gateway | `bootstrap` → `configure_openclaw` (compat fixes) + `start_openclaw_gateway` |
| 4 chat | `prompt` → `_prompt_run` infer |
| 5 dashboard | notebook-only (works because your browser owns the runtime) |

DiffusionGemma is a **reasoning** model (`enable_thinking` is on) — replies include a thinking
trace and take a while; that's expected.

---

### 1 — Install vLLM (nightly cu129) + OpenClaw  *(master: `bootstrap` → install)*

In [1]:
import subprocess, sys
print("nvidia-smi (confirm this is an L4 with ~24 GB) ...", flush=True)
subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True)

print("\nInstalling vLLM (nightly cu129; ~3-5 min) ...", flush=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U", "vllm", "--pre",
                "--extra-index-url", "https://wheels.vllm.ai/nightly/cu129",
                "--extra-index-url", "https://download.pytorch.org/whl/cu129"], check=True)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-U", "huggingface_hub", "hf_transfer"], check=True)

print("Installing OpenClaw (npm-based installer) ...", flush=True)
subprocess.run("curl -fsSL https://openclaw.ai/install.sh | bash -s -- --no-onboard",
               shell=True, check=True)
print("\nInstall complete.")

nvidia-smi (confirm this is an L4 with ~24 GB) ...

Installing vLLM (nightly cu129; ~3-5 min) ...
Installing OpenClaw (npm-based installer) ...

Install complete.


### 2 — Serve DiffusionGemma-26B-A4B-NVFP4 on :8000 + wait for ready  *(master: `bootstrap` → `start_vllm`)*

In [2]:
import subprocess, sys, os, time, glob, shutil, urllib.request
MODEL = "RedHatAI/diffusiongemma-26B-A4B-it-NVFP4"

# Put the CUDA runtime libs from the nvidia-* pip wheels on the loader path (mirrors the bash
# master's start_vllm) so `import vllm._C` finds libcudart.
ld = ":".join(glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib"))
env = dict(os.environ, HF_HUB_ENABLE_HF_TRANSFER="1", VLLM_USE_V2_MODEL_RUNNER="1")
if ld:
    env["LD_LIBRARY_PATH"] = ld + (":" + env["LD_LIBRARY_PATH"] if env.get("LD_LIBRARY_PATH") else "")
# Optionally set HF_TOKEN in the cell above if the checkpoint is gated: env["HF_TOKEN"] = "hf_..."

# vLLM serve args == configs/diffusiongemma_nvfp4.json -> serve_args. The JSON-valued flags are
# passed as single list elements (NO shell), so they don't need the shlex-quoting the bash path did.
vllm = shutil.which("vllm") or "vllm"
serve = [vllm, "serve", MODEL,
    "--trust-remote-code", "--max-num-seqs", "1", "--gpu-memory-utilization", "0.90",
    "--max-model-len", "8192",
    "--hf-overrides", '{"diffusion_sampler":"entropy_bound","diffusion_entropy_bound":0.1}',
    "--default-chat-template-kwargs", '{"enable_thinking":true}',
    "--host", "127.0.0.1", "--port", "8000"]
subprocess.Popen(serve, env=env,
                 stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)

print("Downloading ~13 GB NVFP4 + loading on the L4 (~10-12 min, one time) ...", flush=True)
t0 = time.time(); ready = False
while time.time() - t0 < 1500:
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/v1/models", timeout=4); ready = True; break
    except Exception:
        print(f"   loading {int(time.time()-t0)}s ...", flush=True)
        subprocess.run("tail -1 /content/vllm.log", shell=True); time.sleep(10)
print(("vLLM READY after %ds" % (time.time()-t0)) if ready else
      "TIMEOUT — check /content/vllm.log (OOM? confirm this is a 24 GB L4)")

   loading 0s ...
   loading 10s ...
   loading 20s ...
   loading 30s ...
   loading 40s ...
   loading 50s ...
   loading 60s ...
   loading 70s ...
   loading 80s ...
   loading 90s ...
   loading 100s ...
   loading 110s ...
   loading 120s ...
   loading 130s ...
   loading 140s ...
   loading 150s ...
   loading 160s ...
   loading 170s ...
   loading 180s ...
   loading 190s ...
   loading 200s ...
   loading 210s ...
   loading 220s ...
   loading 230s ...
   loading 240s ...
   loading 250s ...
   loading 260s ...
   loading 270s ...
   loading 280s ...
   loading 290s ...
   loading 300s ...
   loading 310s ...
   loading 320s ...
   loading 330s ...
   loading 340s ...
   loading 350s ...
   loading 360s ...
vLLM READY after 370s


### 3 — Onboard OpenClaw against :8000 + start gateway  *(master: `bootstrap` → `configure_openclaw` + gateway)*

In [3]:
import subprocess, shutil, os, time
# Resolve openclaw by ABSOLUTE path -> never 'openclaw: command not found' (npm symlinks it into
# the global bin). Provider id 'vllm' matches the bash harness + the chat/dashboard cells.
OPENCLAW = shutil.which("openclaw") or "/usr/bin/openclaw"
os.environ.setdefault("OPENCLAW_GATEWAY_TOKEN", "dg-local-token")
def oc(args): return subprocess.run([OPENCLAW] + args, capture_output=True, text=True)
print("openclaw:", OPENCLAW)

ob = oc(["onboard", "--non-interactive", "--accept-risk", "--mode", "local",
    "--auth-choice", "custom-api-key", "--custom-provider-id", "vllm",
    "--custom-base-url", "http://127.0.0.1:8000/v1",
    "--custom-model-id", "RedHatAI/diffusiongemma-26B-A4B-it-NVFP4", "--custom-compatibility", "openai",
    "--custom-api-key", "vllm-local", "--custom-text-input",
    "--gateway-port", "18789", "--gateway-bind", "loopback", "--gateway-auth", "token",
    "--gateway-token-ref-env", "OPENCLAW_GATEWAY_TOKEN",
    "--skip-daemon", "--skip-skills", "--skip-channels", "--skip-health", "--skip-ui", "--json"])
print("onboard rc =", ob.returncode)

# Infer fixes == configs/diffusiongemma_nvfp4.json -> openclaw.compat (string content; tools off;
# maxTokens < max-model-len so the completion can't overflow the window -> empty reply).
for k, v in [("compat.requiresStringContent", "true"), ("compat.supportsTools", "false"),
             ("maxTokens", "512"), ("contextWindow", "4096")]:
    oc(["config", "set", f"models.providers.vllm.models[0].{k}", v])

# Gateway runs in-process (lives while the tab is open) — needed for the inline dashboard (cell 5).
subprocess.Popen([OPENCLAW, "gateway", "run"],
    stdout=open("/content/gateway.log", "w"), stderr=subprocess.STDOUT)
time.sleep(12)
print("gateway started on 127.0.0.1:18789")

openclaw: /usr/bin/openclaw
onboard rc = 0
gateway started on 127.0.0.1:18789


### 4 — 💬 Chat with DiffusionGemma  *(master: `prompt` → `_prompt_run`)*

Edit `MESSAGE` and re-run this cell per turn. Uses **direct** infer (no gateway needed, the robust
path). DiffusionGemma reasons before answering, so a reply takes a while and includes a thinking
trace — that's the model, not a hang.

In [4]:
MESSAGE   = "Hello! In one sentence, who are you?"   # <- edit me; re-run for each turn
MODEL_REF = "vllm/RedHatAI/diffusiongemma-26B-A4B-it-NVFP4"

import subprocess, json, shutil
OPENCLAW = shutil.which("openclaw") or "/usr/bin/openclaw"
r = subprocess.run([OPENCLAW, "infer", "model", "run", "--model", MODEL_REF,
                    "--prompt", MESSAGE, "--json"], capture_output=True, text=True)
try:
    print(json.loads(r.stdout)["outputs"][0]["text"])
except Exception:
    print(r.stdout or r.stderr)

thought
*   Question: "Hello! In one sentence, who are you?"
    *   Constraint: Answer in one sentence.

    *   Name: Gemma 4.
    *   Developer: Google DeepMind.
    *   Nature: Large Language Model.
    *   Type: Open weights model.

    *   "I am Gemma 4, a large language model with open weights developed by Google DeepMind."

    *   Does it mention Gemma 4? Yes.
    *   Does it mention Google DeepMind? Yes.
    *   Is it one sentence? Yes.
    *   Does it mention the nature/type? Yes.I am Gemma 4, a large language model with open weights developed by Google DeepMind.


### 5 — (Optional) OpenClaw Control dashboard, inline — no tunnel

Works **only because your browser owns this runtime**, so `serve_kernel_port_as_iframe` mints a
Google-authenticated proxy to the gateway port 18789 — no public tunnel. (This is exactly why
the dashboard is NOT reachable when the headless CLI manages the VM.) Needs the gateway from cell 3.

In [5]:
from google.colab import output
import os
print("If the dashboard asks for a token, paste:", os.environ.get("OPENCLAW_GATEWAY_TOKEN", "dg-local-token"))
output.serve_kernel_port_as_iframe(18789, path="/", height="720")

If the dashboard asks for a token, paste: dg-local-token


<IPython.core.display.Javascript object>